In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

from armored.models import *
from armored.preprocessing import *

from sklearn.model_selection import KFold

import itertools

from tqdm import tqdm

/home/jcthompson5@ad.wisc.edu/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# data with initial and end point measurements
df_exp0 = pd.read_csv("data/exp0/exp0_metabolites.csv")
df_exp1 = pd.read_csv("data/exp1/exp1_metabolites.csv")
df_exp2 = pd.read_csv("data/exp2/exp2_metabolites.csv")
df_exp3 = pd.read_csv("data/exp3/exp3_metabolites.csv")

In [3]:
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']
metabolites = ['pH', 'Lactate', 'Butyrate', 'Acetate']

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

array(['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs',
       'CHabs', 'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs',
       'RIabs', 'pH', 'Lactate', 'Butyrate', 'Acetate', 'AcGum', 'ArGal',
       'Inulin', 'Pectin', 'Starch', 'Xylan'], dtype='<U8')

In [4]:
# separate monoculture data (monoculture always trained on, never used as held-out data)
df_mono = []
df_comm = []

for exp, df_exp in df_exp0.groupby("Experiments"):
    
    # check if monoculture or not
    if sum(df_exp.iloc[df_exp.Time.values==0][species].values[0] > 0) > 1:
        df_comm.append(df_exp)
    else:
        df_mono.append(df_exp)
        
# concatenate dfs
df_mono = pd.concat(df_mono)
df_exp0 = pd.concat(df_comm)

In [5]:
# df for training 
df = pd.concat((df_exp0, df_exp1, df_exp2, df_exp3))

In [6]:
all_treatments = df.Experiments.values
unique_treatments = np.unique(all_treatments) 

n_splits = 20
kf = KFold(n_splits=n_splits, shuffle=True, random_state=21)
k_fold_df = pd.DataFrame()
for i, (train_index, test_index) in enumerate(kf.split(unique_treatments)):

    # split into train and test
    train_index = np.in1d(all_treatments, unique_treatments[train_index])
    test_index  = np.in1d(all_treatments, unique_treatments[test_index])
    train_df = df.iloc[train_index].copy()
    test_df  = df.iloc[test_index].copy() 
    
    # add monoculture data
    train_df = pd.concat((df_mono, train_df))
    
    # scale data 
    scaler = MinMaxScaler(observed, system_variables)
    scaler.fit(train_df)
    train_df_scaled = scaler.transform(train_df.copy())
    test_df_scaled = scaler.transform(test_df.copy())
    
    # format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
    train_data = format_data(train_df, species, metabolites, controls, observed=observed)
    train_data_scaled = format_data(train_df_scaled, species, metabolites, controls, observed=observed)
    test_data = format_data(test_df, species, metabolites, controls, observed=observed)
    test_data_scaled = format_data(test_df_scaled, species, metabolites, controls, observed=observed)

    # instantiate model
    lr = LR2(n_species=len(species), 
             n_metabolites=len(metabolites), 
             n_controls=len(controls))
    
    # fit model
    lr.fit(train_data_scaled, 
           alpha_0=0., alpha_1=1.,
           evd_tol=1e-3)

    # make predictions
    predictions = scaler.inverse_transform(lr.predict(test_data_scaled))

    # save predictions
    pred_df = pd.DataFrame()
    for (T, X, U, Y, exp_names), (_, preds, stdvs, exp_names_pred) in zip(test_data, predictions):

        # save species predictions for each experimental condition
        for i, exp_name in enumerate(exp_names):
            # init dataframe
            pred_df_exp = pd.DataFrame()

            # insert exp name
            pred_df_exp["Experiments"] = [exp_name]*len(T[i])
            pred_df_exp["Time"] = T[i]

            for j, s in enumerate(observed):
                pred_df_exp[s + " true"] = Y[i,:,j]
                pred_df_exp[s + " pred"] = preds[i,:,j]
                pred_df_exp[s + " stdv"] = stdvs[i,:,j]

            # append to test prediction dataframe
            pred_df = pd.concat((pred_df, pred_df_exp))
    k_fold_df = pd.concat((k_fold_df, pred_df))
    k_fold_df.to_csv(f"kfold/LR2_20_fold_dtl_3.csv", index=False)

Total measurements: 25536, Number of parameters: 6194, Initial regularization: 0.00e+00
Loss: 2600.170, Residuals: 0.00782
Loss: 1463.961, Residuals: -0.01617
Loss: 989.702, Residuals: -0.01543
Loss: 946.827, Residuals: -0.00370
Loss: 869.508, Residuals: -0.00263
Loss: 744.839, Residuals: -0.00177
Loss: 578.037, Residuals: -0.00193
Loss: 556.732, Residuals: 0.00124
Loss: 529.752, Residuals: -0.00041
Loss: 526.921, Residuals: 0.00129
Loss: 526.059, Residuals: 0.00122
Updating precision...
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
Evidence 5655.893
Loss: 2061.342, Residuals: 0.00017
Loss: 1966.349, Residuals: 0.00046
L

2024-11-04 17:52:03.960737: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 25457, Number of parameters: 6194, Initial regularization: 0.00e+00
Loss: 2627.104, Residuals: 0.00847
Loss: 1488.161, Residuals: -0.01681
Loss: 996.190, Residuals: -0.01587
Loss: 951.499, Residuals: -0.00391
Loss: 877.811, Residuals: -0.00333
Loss: 753.471, Residuals: -0.00210
Loss: 609.052, Residuals: -0.00093
Loss: 515.949, Residuals: -0.00043
Loss: 503.149, Residuals: 0.00094
Loss: 501.558, Residuals: 0.00123
Loss: 498.195, Residuals: 0.00103
Loss: 491.993, Residuals: 0.00071
Loss: 491.167, Residuals: 0.00082
Loss: 490.573, Residuals: 0.00081
Updating precision...
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding re

2024-11-04 19:19:42.798617: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 25581, Number of parameters: 6194, Initial regularization: 0.00e+00
Loss: 2644.137, Residuals: 0.00776
Loss: 1489.761, Residuals: -0.01761
Loss: 994.772, Residuals: -0.01494
Loss: 952.182, Residuals: -0.00378
Loss: 875.247, Residuals: -0.00297
Loss: 747.110, Residuals: -0.00231
Loss: 588.197, Residuals: -0.00065
Loss: 569.060, Residuals: 0.00076
Loss: 532.894, Residuals: 0.00071
Loss: 476.517, Residuals: -0.00004
Loss: 472.280, Residuals: 0.00192
Loss: 465.837, Residuals: 0.00153
Loss: 465.108, Residuals: 0.00153
Updating precision...
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization

...

2024-11-05 21:38:18.309318: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 25678, Number of parameters: 6194, Initial regularization: 0.00e+00
Loss: 2667.702, Residuals: 0.00857
Loss: 1497.962, Residuals: -0.01644
Loss: 1003.022, Residuals: -0.01558
Loss: 957.399, Residuals: -0.00391
Loss: 883.405, Residuals: -0.00329
Loss: 758.441, Residuals: -0.00202
Loss: 613.031, Residuals: -0.00077
Loss: 521.414, Residuals: -0.00064
Loss: 506.345, Residuals: 0.00105
Loss: 492.568, Residuals: 0.00047
Loss: 489.920, Residuals: 0.00056
Loss: 489.179, Residuals: 0.00057
Updating precision...
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
Evidence 5745.333
Loss: 1970.300, Residuals: 0.00013
L

2024-11-05 23:03:48.358420: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed


Total measurements: 25476, Number of parameters: 6194, Initial regularization: 0.00e+00
Loss: 2602.288, Residuals: 0.00747
Loss: 1471.168, Residuals: -0.01702
Loss: 987.693, Residuals: -0.01480
Loss: 947.672, Residuals: -0.00350
Loss: 880.312, Residuals: -0.00306
Loss: 764.743, Residuals: -0.00219
Loss: 614.816, Residuals: -0.00105
Loss: 521.036, Residuals: -0.00002
Loss: 507.444, Residuals: 0.00069
Loss: 506.351, Residuals: 0.00156
Loss: 464.843, Residuals: 0.00082
Loss: 459.459, Residuals: 0.00114
Loss: 458.397, Residuals: 0.00123
Loss: 457.937, Residuals: 0.00122
Updating precision...
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding regularization
adding re

2024-11-06 00:36:13.947554: W external/xla/xla/service/cpu/onednn_matmul.cc:172] [Perf]: MatMul reference implementation being executed
